# OPENAI LLM Baseline Code

# 모델 불러오기

In [7]:
from dotenv import load_dotenv 
load_dotenv()

True

In [8]:
from openai import OpenAI

client = OpenAI()

# 요청

In [10]:
def chat(system_prompt, user):
    response = client.chat.completions.create(
    model="gpt-4.1-nano",
    messages=[
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": user
        }
    ],
    temperature = 0.7
    )

    answer = response.choices[0].message.content
    return answer

def chat5(system_prompt, user, verbosity = 'medium'):
    response = client.chat.completions.create(
    model="gpt-5-nano",
    messages=[
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": user
        }
    ],
    verbosity = verbosity # low: 간결, 핵심 위주, medium: 적당한 설명과 예시, high: 자세하고 풍부한 설명
    )

    answer = response.choices[0].message.content
    return answer

In [ ]:
system_prompt = """
당신은 약품 성분에 대해서 잘 알고 있는 약사 AI 입니다.
사용자가 가지고 있는 병과, 주의해야하는 알약, 주의해야하는 성분을 제공받고
현재 사용자가 물어보는 알약을 먹어도 되는지 안되는지 판단해야합니다.

[판단 방법]
1단계: 사용자의 '주의 성분'이 약품의 [성분] 또는 [첨가제]에 포함되는가?
2단계: 사용자의 '주의 약품'이 해당 약품이 포함되는가?
3단계: 사용자의 '나의 병명'에 해당하는 질병이 있을 경우 해당 약품을 섭취해도 되는가?
4단계: 사용자의 '복용 중인 약'과의 상호작용 가능성이 있는가?

[입력 정보]
섭취 하려는 약 : 사용자가 섭취하려고 하는 약 이름
나의 병명: 내가 가지고 있는 질병 명들
주의 약품 : 내가 주의해야하는 약품 명들 (내가 주의해야하는 약품과 비슷한 성분이 있을 경우 미섭취 권장)
주의 성분 : 내가 주의해야하는 성분 명등 (주의해야하는 성분이 있을 경우 미섭취 권장)
복용 중인 약 : 내가 복용 중인 약

[사전 정보]
제품명명 : 게루삼정Gelusam Tab.,
성분/함량 : "['Calcium Carbonate P.P.T.\u3000침강탄산칼슘\u3000100mg','Dried Aluminum Hydroxide Gel\u3000건조수산화알루미늄겔\u3000200mg', 'Magnesium Carbonate\u3000탄산마그네슘\u300050mg', 'Sodium Bicarbonate\u3000탄산수소나트륨\u300050mg']",
성분: "['스테아르산마그네슘', '카르복시메틸셀룰로오스칼슘']",
정제,흰색의 원형정제,
"['소화기계질환', '소화성궤양 치료제', '제산제', '흡수성', '소화기계질환', '소화성궤양 치료제', '제산제', '비흡수성', '소화기계질환', '소화성궤양 치료제', '제산제', '흡수성', '소화기계질환', '소화성궤양 치료제', '제산제', '흡수성']",
"기밀용기, 실온(1~30℃)보관",
"위산과다, 속쓰림, 위부불쾌감, 위부팽만감, 체함, 구역, 구토, 위통, 신트림.",
"만 15세 이상 및 성인: 1회 2정, 1일 3회",
"1. 다음과 같은 사람은 이 약을 복용하지 말 것. 1) 투석요법을 받고 있는 환자 2) 만 7세 이하의 어린이 2. 이 약을 복용하는 동안 다음의 약을 복용하지 말 것. 1) 테트라사이클린계 항생제 3. 다음과 같은 사람은 이 약 복용하기 전에 의사, 치과의사, 약사와 상의할 것. 1) 신장장애 2) 다른 약물을 복용하고 있는 사람 4. 다음과 같은 사람( 경우) 이 약의 복용을 즉각 중지하고 의사, 치과의사, 약사와 상의할 것. 상담시 이 첨부문서를 소지할 것. 1) 이 약을 복용하는 동안 변비 또는 설사의 증상이 나타날 경우 2) 2주정도 투여하여도 증상의 개선이 없을 경우 5. 기타 이 약의 복용시 주의할 사항 1) 정해진 용법ㆍ용량을 잘 지킬 것. 2) 어린이 투여할 경우에는 보호자의 지도 감독하에 투여할 것. 3) 나트륨 제한 식이를 하는 사람. 6. 저장상의 주의사항 1) 어린이의 손에 닿지 않는 곳에 보관할 것. 2) 직사일광을 피하고 되도록 습기가 적은 서늘한 곳에 보관할 것. 3) 오용을 막고 품질의 보존을 위하여 다른 용기에 바꾸어 넣지 말 것.",K-000059,일반의약품,게루삼정 200T,234.0,200T,게루삼정,inf

[출력 형식]
{
    "name" : "제품명",
    "effect" : "이 약의 효능",
    "isUsable" : "섭취 가능 여부를 Bool 값으로",
    "unusable_reason" : "섭취가 불가능한 이유를 설명합니다, 섭취 가능한 경우 빈칸으로 제공합니다",
    "cautionary_ingredients" : "섭취 불가능한 이유가 되는 주의 성분을 리스트로 제공합니다, 섭취 가능한경우 일반적인 주의사항을 제공합니다"
}}
"""

In [5]:
drug_name = "게루삼정"
disease_names = ""
cautionary_medications = ""
cautionary_ingredients = "스테아르산마그네슘"
taking_drug = ""

In [6]:
user = f"""
섭취 하려는 약 : {drug_name}
나의 병명: {disease_names}
주의 약품 : {cautionary_medications}
주의 성분 : {cautionary_ingredients}
섭취 중인 약 : {taking_drug}
"""

In [7]:
answer = chat(system_prompt, user)
print(answer)

{
    "name": "게루삼정",
    "effect": "이 약은 소화기계 질환, 소화성 궤양 치료제, 제산제로서 위산과다, 속쓰림, 위부불쾌감, 위부팽만감, 체함, 구역, 구토, 위통, 신트림 등의 증상을 완화하는 데 사용됩니다.",
    "isUsable": true,
    "unusable_reason": "",
    "cautionary_ingredients": "일반적인 주의사항을 참고하시기 바라며, 현재 주의 성분인 스테아르산마그네슘을 포함하고 있어 특별한 금기사항은 없습니다. 다만, 특정 성분에 과민반응이 있거나 의사의 지시 없이 복용을 중단하지 마세요."
}


---

In [13]:
import json
facts = {
    '제품명': '타이레놀정', 
    '성분/함량': [['아세트아미노펜(USP)', '50Mg']], 
    '첨가물': ['옥수수전분', '전분', '셀룰로오스', '스테아르산마그네슘'], 
    '제형/성상': '백색의 장방형필름코팅정제', 
    'KPIC/ATC': '없음', 
    '구분': '없음', 
    '효능': ['감기로인한발열및동통(통 증)', '두통', '신경통', '근육통', '월경통', '염좌통(뻔통증)', '치통', '관절통', '류마티양동통(통증이상 또는그병력)'], 
    '용법': ['12세이상 소아및성인 1회 1-2정씩 1일 3-4회(4-6시간마다)최대 8정(4a)를 초과하여 복용하지 말 것'], 
    '주의사항': ['삼환계항우 울제·알코올을복용한사람', '심한심장기능저하환자', '아스피린천식(비스테로있는환자)', '바르비탈계약물', '이드성소염(항염)제에의한천식발작유발', '유효성분 1정중아세트아 미노펜(USP)…50Mg', '효능효과감기로인한발열및동통(통증)두통,신경통근육통,월경통,염좌통(뻔통증)치통,관절통,류마티양동통(통증이상 또는그병력)', '출혈경향이있는환자', '간장애 또는그병력', '신장(콩팥)장애 또는그병력', '소화성궤양의병력', '혈액', '복용 시 주의할사항', '복용 후속환자', '과민증의병력있는환자', '기관지천식환자', '고령자(노인)', '임부/수유부', '파린장기복용환자(혈소판기능이상)', '심장기능이상환자', '복용 하여서는안됨', '아세트아미노펜을포함하는다른제품과함께', '약사와상의할것', '이약투 여후피부발진이나다른이약', '일일최대용량(4000Mg)을초과하여복용시간손상', '매일세잔 이상정기적술복용시주의', '과민반응', '즉시복용중단', '아세트아미노펜함유', '최다손 이닿지않는곳에보관', '치과의사약사와상의할것']}
package_data = json.dumps(facts, ensure_ascii=False)

import ast
raw_data = ['{"C-Code": "K-004378", "구분": "일반의약품", "제품명": "타이레놀정500mg(성인용)", "복지부분류코드": 114.0, "volume": "500mg", "name": "타이레놀정", "vol_value": 500.0, "vol_unit": "mg", "성분/함량": "[\'Acetaminophen 아세트아미노펜 500mg\']", "첨가물": "[\'분말셀룰로오스\', \'스테아르산마그네슘\', \'오파드라이흰색(YS-1-7027)\', \'옥수수전분\', \'전분글리콜산나트륨\', \'전호화전분\', \'카르나우바납\']", "제형": "정제", "성상": "흰색의 장방형 필름코팅정제", "KPIC/ATC": "[\'통증 질환\', \'비마약성 진통제\', \'중추성 진통제\', \'p-aminophenol 유도체\', \'N02BE01\']", "구분(전문/일반)": "일반", "효능": "1. 주효능·효과 1. 주효능·효과 감기로 인한 발열 및 동통(통증), 두통, 신경통, 근육통, 월경통, 염좌통(삔 통증) 2. 다음 질환에도 사용할 수 있다. 치통, 관절통, 류마티양 동통(통증)", "용법": "만 12세 이상 소아 및 성인: 만 12세 이상 소아 및 성인: 1회 1~2정씩 1일 3-4회 (4-6시간 마다) 필요시 복용한다. 1일 최대 4그램 (8정)을 초과하여 복용하지 않는다. 이 약은 가능한 최단기간동안 최소 유효용량으로 복용한다.", "주의사항": "1. 경고 1) 매일 세잔 이상 정기적으로 술을 마시는 사람이 이 약이나 다른 해열 진통제를 복용해야 할 경우 반드시 의사 또는 약사와 상의해야 한다. 이러한 사람이 이 약을 복용하면 간손상이 유발될 수 있다. 2) 아세트아미노펜을 복용한 환자에서 매우 드물게 급성 전신성 발진성 농포증(급성 전신성 발진성 고름물집증)(AGEP), 스티븐스 - 존슨 증후군(SJS), 독성 표피 괴사용해(TEN)와 같은 중대한 피부 반응이 보고되었 고, 이러한 중대한 피부반응은 치명적일 수 있다. 따라서 이러한 중대한 피부반응의 징후에 대하여 환자들에게 충분히 알리고, 이 약 투여 후 피부발진이나 다른 과민반응의 징후가 나타나면 즉시 복용을 중단하도록 하여야 한다. 3) 이 약은 아세트아미노펜을 함유하고 있다.  아세트아미노펜으로 일일 최대 용량(4,000mg)을 초과할 경우 간손상을 일으킬 수 있으므로 이 약을 일일 최대 용량(4000mg)을 초과하여 복용하여서는 아니되며, 아세트아미노펜을 포함하는 다른 제품과 함께 복용하여서는 안 된다. 2. 다음과 같은 사람은 이 약을 복용하지 말  것 1) 이 약에 과민증 환자 2) 소화성궤양 환자 3) 심한 혈액 이상 환자 4) 심한 간장애 환자 5) 심한 신장(콩팥)장애 환자 6) 심한 심장 기능저하 환자 7) 아스피린 천식(비스테로이드성 소염(항염)제에 의한 천식발작 유발) 또는 그 병력이 있는 환자 8) 다음의 약물을 복용한 환자 : 바르비탈계 약물, 삼환계 항우울제 9) 알코올을 복용한 사람 3. 다음과 같은 사람은 이약을 복용하기 전에 의사, 치과의사, 약사 와 상의할 것 1) 간장애 또는 그 병력이 있는 환자 2) 신장(콩팥)장애 또는 그 병력이 있는 환자 3) 소화성궤양의 병력이 있는 환자 4) 혈액이상 또는 그 병력이 있는 환자 5) 출혈경향이 있는 환자(혈소판기능이상이 나타날 수 있다.) 6) 심장기능이상이 있는 환자 7) 과민증의 병력이 있는 환자 8) 기관지 천식 환자 9) 고령자(노인) 10) 임부 또는 수유부 11) 다음의 약물을 복용하는 환자 : 와파린, 플루클록사실린 12) 글루타치온 보유량이 낮은 상태 환자 4. 다음과 같을 경우 이 약의 복용을 즉각 중지하고 의사, 치과의사, 약사와 상의할 것. 상담시 가능한 한 이 첨부문서를 소지할 것 1) 쇽: 쇽, 아나필락시양 증상 (과민성유사증상 : 호흡곤란, 온몸이 붉어짐, 혈관부기, 두드러기  등), 천식발작 2) 혈액: 혈소판 감소, 과립구감소, 용혈성(적혈구 파괴성)빈혈, 메트헤모글로빈혈증, 혈소판기능 저하(출혈시간 연장), 청색증 3) 과민증: 과민증상(얼굴부기, 호흡곤란, 땀이 남, 저혈압, 쇽) 4) 소화기: 구역, 구토, 식욕부진, 장기복용시 위장출혈, 소화성궤 양, 천공(뚫림) 등의 위장관계 이상반응 5) 피부: 발진, 알레르기 반응, 피부점막안 증후군(스티븐스-존슨 증후군), 중독성표피괴사용해( 리엘 증후군) 6) 기타: 장기투여시 만성간괴사, 급성췌장(이자)염, 만성간염, 신장(콩팥)독성 7) 과량투여: 간장, 신장(콩팥), 심근의 괴 사 8) 이 약에 대해 시판 후 조사에서 보고된 추가적 이상반응은 아래 표와 같다. 발현빈도는 매우 흔히 ≥1/10, 흔히 ≥1/100 및 <1/10, 흔하지 않게 ≥1/1,000 및 <1/100, 드물게 ≥1/10,000 및 <1/1,000, 매우 드물게 <1/10,000 이다. 면역계 장애 매우 드물게 : 아나필락시스 반응, 과민증 피부 및 피하(피부밑)조직 장애 매우 드물게 : 두드러기, 소양성(가려움) 발진, 발진, 고정발진 표. 자발적 보고율로부터 추정한 빈도에 따른 이 약의 시판후 경험에서 밝혀진 이상반응 9) 국내 부작용 보고자료의 분석·평가에 따라 다음의 이상반응을 추가한다. ●  간담도계: AST 상승, ALT 상승 ● 피부: 고정발진 5. 기타 이 약을 복용시 주의할 사항 1) 일반적주의 (1) 과민증상을 예측하기 위해 충분 한 상담을 받아야 한다. (2) 소염(항염)진통제에 의한 치료는 원인요법이 아닌 대증요법(증상별로 치료하는 방법)이다. (3) 만성질환에 사용하는 경우에는 다음 사항을 고려한다. 가. 장기복용하는 경우 정기적인 임상검사(요검사, 혈액검사, 간기능검사 등)를 받고 이상이 있을 경우 감량(줄임), 복용중지 등의 적절한 조치를 해야 한다. 나. 약물요법 이외의 치료법도 고려한다.(4) 급성질환에 사용하는 경우에는  다음 사항을 고려한다. 가. 급성통증 및 발열의 정도를 고려하여 복용한다. 나. 원칙적으로 동일한 약물의 장기복용은 피한다. 다. 원인요법이 있는 경우에는 실시한다.(5) 소아 및 고령자(노인)는 최소 필요량을 복용하고 이상반응에 유의한다. 과도한 체온강하, 허탈, 사지냉 각 등이 나타날 수 있으므로, 특히 고열을 수반하는 소아 및 고령자(노인) 또는 소모성 질환 환자의 경우 복용 후의 상태를 충분히 살펴야한다. (6) 다른 소염(항염)진통제와 함께 복용하는 것은 피한다. (7) 의사 또는 약사의 지시없이 통증에 10일 이상(성인) 또는 5일 이상( 소아) 복용하지 않고 발열에 3일 이상 복용하지 않는다. 통증이나 발열 증상이 지속되거나 악화될 경우, 또는 새로운 증상이 나타날 경우 의사 또는 약사와 상의한다. (8) 이 약 복용시 감염증을 겉으로 나타나지 않게 할 수 있으므로 감염증이 합병된 환자의 경우에 의사처방에 따라 적절한 항균제를 함께 복용해야 한다. 2) 과량투여시의 처치 이 약을 과량복용시 어떠한 명백한 증상이나 징후가 없더라도 신속하게 의학적 처치를 받아야 한다. 10~12시간 이내에 N-아세틸시스테인 정맥주사를 투여받거나 메치오닌을 경구복용하여 간을 보호해야한다. 6. 저장상의 주의사항 1) 밀폐용기, 실온(1~30℃)보관 2) 어린이의 손이 닿지 않는 곳에 보관한다. 3) 의약품을 원래의 용기에서 꺼내어 다른 용기에 보관하는 것은 의약품의 오용(잘못 사용)에 따른 사고 발생이나 의약품 품질 저하의 원인이 될 수 있으므로 원래의 용기에 넣고  꼭 닫아 보관한다.", "저장방법": "밀폐용기, 실온보관(1-30℃)", "name_ko": "타이레놀정", "name_en": "mg Tylenol Tab"}', '{"C-Code": "K-004799", "구분": "일반의약품", "제품명": "타이레놀정160mg", "복지부분류코드": 114.0, "volume": "160mg", "name": "타이레놀정", "vol_value": 160.0, "vol_unit": "mg", "성분/함량": "[\'Acetaminophen 아세트아미노펜 500mg\']", "첨가물": "[\'분말셀룰로오스\', \'스테아르산마그네슘\', \'오파드라이흰색(YS-1-7027)\', \'옥수수전분\', \'전분글리콜산나트륨\', \'전호화전분\', \'카르나우바납\']", "제형": "정제", "성상": "흰색의 장방형 필름코팅정제", "KPIC/ATC": "[\'통증 질환\', \'비마약성 진통제\', \'중추성 진통제\', \'p-aminophenol 유도체\', \'N02BE01\']", "구분(전문/일반)": "일반", "효능": "1. 주효능·효과 1. 주효능·효과 감기로 인한 발열 및 동통(통 증), 두통, 신경통, 근육통, 월경통, 염좌통(삔 통증) 2. 다음 질환에도 사용할 수 있다. 치통, 관절통, 류마티양 동통(통증)", "용법": "만 12세 이상 소아 및 성인: 만 12세 이상 소아 및 성인: 1회 1~2정씩 1일 3-4회 (4-6시간 마다) 필요시 복용한다. 1일 최대 4그램 (8정) 을 초과하여 복용하지 않는다. 이 약은 가능한 최단기간동안 최소 유효용량으로 복용한다.", "주의사항": "1. 경고 1) 매일 세잔 이상 정 기적으로 술을 마시는 사람이 이 약이나 다른 해열 진통제를 복용해야 할 경우 반드시 의사 또는 약사와 상의해야 한다. 이러한 사람이 이 약을 복용하면 간손상이 유발될 수 있다. 2) 아세트아미노펜을 복용한 환자에서 매우 드물게 급성 전신성 발진성 농포증(급성 전신성 발 진성 고름물집증)(AGEP), 스티븐스 - 존슨 증후군(SJS), 독성 표피 괴사용해(TEN)와 같은 중대한 피부 반응이 보고되었고, 이러한 중대한 피부반응은 치명적일 수 있다. 따라서 이러한 중대한 피부반응의 징후에 대하여 환자들에게 충분히 알리고, 이 약 투여 후 피부발진이나  다른 과민반응의 징후가 나타나면 즉시 복용을 중단하도록 하여야 한다. 3) 이 약은 아세트아미노펜을 함유하고 있다. 아세트아미노펜으로 일일 최대 용량(4,000mg)을 초과할 경우 간손상을 일으킬 수 있으므로 이 약을 일일 최대 용량(4000mg)을 초과하여 복용하여서는 아니되 며, 아세트아미노펜을 포함하는 다른 제품과 함께 복용하여서는 안 된다. 2. 다음과 같은 사람은 이 약을 복용하지 말 것 1) 이 약에 과민증 환자 2) 소화성궤양 환자 3) 심한 혈액 이상 환자 4) 심한 간장애 환자 5) 심한 신장(콩팥)장애 환자 6) 심한 심장기능저하 환자 7) 아스피린 천식(비스테로이드성 소염(항염)제에 의한 천식발작 유발) 또는 그 병력이 있는 환자 8) 다음의 약물을 복용한 환자 : 바르비탈계 약물, 삼환계 항우울제 9) 알코올을 복용한 사람 3. 다음과 같은 사람은 이약을 복용하기 전에 의사, 치과의사, 약사와 상의할 것 1) 간장애 또는 그 병력이 있는 환자 2) 신장(콩팥)장애 또는 그 병력이 있는 환자 3) 소화성궤양의 병력이 있는 환자 4) 혈액이상 또는 그 병력 이 있는 환자 5) 출혈경향이 있는 환자(혈소판기능이상이 나타날 수 있다.) 6) 심장기능이상이 있는 환자 7) 과민증의 병력이 있는 환자 8) 기관지 천식 환자 9) 고령자(노인) 10) 임부 또는 수유부 11) 다음의 약물을 복용하는 환자 : 와파린, 플루클록사실린 12) 글루타치온  보유량이 낮은 상태 환자 4. 다음과 같을 경우 이 약의 복용을 즉각 중지하고 의사, 치과의사, 약사와 상의할 것. 상담시 가능한 한 이 첨부문서를 소지할 것 1) 쇽: 쇽, 아나필락시양 증상 (과민성유사증상 : 호흡곤란, 온몸이 붉어짐, 혈관부기, 두드러기 등), 천식발작 2) 혈액: 혈소판 감소, 과립구감소, 용혈성(적혈구 파괴성)빈혈, 메트헤모글로빈혈증, 혈소판기능 저하(출혈시간 연장), 청색증 3) 과민증: 과 민증상(얼굴부기, 호흡곤란, 땀이 남, 저혈압, 쇽) 4) 소화기: 구역, 구토, 식욕부진, 장기복용시 위장출혈, 소화성궤양, 천공(뚫림) 등의 위장관계 이상반응 5) 피부: 발진, 알레르기 반응, 피부점막안 증후군(스티븐스-존슨 증후군), 중독성표피괴사용해(리엘 증후군) 6) 기타: 장기투여시 만성간괴사, 급성췌장(이자)염, 만성간염, 신장(콩팥)독성 7) 과량투여: 간장, 신장(콩팥), 심근의 괴사 8) 이 약에 대해 시판 후 조사에서 보고된 추가적 이상반응은 아래 표와 같다. 발현빈도는 매우 흔히 ≥1/10, 흔히 ≥1/100 및 <1/10, 흔하지 않게 ≥1/1,000 및 <1/100, 드물게 ≥1/10,000 및 <1/1,000, 매우 드물게 <1/10,000 이다. 면역계 장애 매우 드물게 : 아나필락시스 반응, 과민증 피부 및 피하(피부밑)조직 장애 매우 드물게 : 두드러기, 소양성(가려움) 발진, 발진, 고정발진 표. 자발적 보고율로부터 추정한 빈도에 따른 이 약 의 시판후 경험에서 밝혀진 이상반응 9) 국내 부작용 보고자료의 분석·평가에 따라 다음의 이상반응을 추가한다. ● 간담도계: AST 상승, ALT 상승 ● 피부: 고정발진 5. 기타 이 약을 복용시 주의할 사항 1) 일반적주의 (1) 과민증상을 예측하기 위해 충분한 상담을 받아야 한다. (2) 소염(항염)진통제에 의한 치료는 원인요법이 아닌 대증요법(증상별로 치료하는 방법)이다. (3) 만성질환에 사용하는 경우에는 다음  사항을 고려한다. 가. 장기복용하는 경우 정기적인 임상검사(요검사, 혈액검사, 간기능검사 등)를 받고 이상이 있을 경우 감량(줄임), 복 용중지 등의 적절한 조치를 해야 한다. 나. 약물요법 이외의 치료법도 고려한다.(4) 급성질환에 사용하는 경우에는 다음 사항을 고려한다. 가. 급성통증 및 발열의 정도를 고려하여 복용한다. 나. 원칙적으로 동일한 약물의 장기복용은 피한다. 다. 원인요법이 있는 경우에는 실시한다.(5) 소아 및 고령자(노인)는 최소 필요량을 복용하고 이상반응에 유의한다. 과도한 체온강하, 허탈, 사지냉각 등이 나타날 수 있으므로, 특히 고열을 수반하는 소아 및 고령자(노인) 또는 소모성 질환 환자의 경우 복용 후의 상태를 충분히 살펴야한다. (6) 다른 소염(항염)진통제와 함께 복용하는 것은 피한다. (7) 의사 또는 약사의 지시없이 통증에 10일 이상(성인) 또는 5일 이상(소아) 복용하지 않고 발 열에 3일 이상 복용하지 않는다. 통증이나 발열 증상이 지속되거나 악화될 경우, 또는 새로운 증상이 나타날 경우 의사 또는 약사와 상의 한다. (8) 이 약 복용시 감염증을 겉으로 나타나지 않게 할 수 있으므로 감염증이 합병된 환자의 경우에 의사처방에 따라 적절한 항균제를 함께 복용해야 한다. 2) 과량투여시의 처치 이 약을 과량복용시 어떠한 명백한 증상이나 징후가 없더라도 신속하게 의학적 처치를 받아야 한다. 10~12시간 이내에 N-아세틸시스테인 정맥주사를 투여받거나 메치오닌을 경구복용하여 간을 보호해야한다. 6. 저장상의 주의사항 1) 밀폐용기, 실온(1~30℃)보관 2) 어린이의 손이 닿지 않는 곳에 보관한다. 3) 의약품을 원래의 용기에서 꺼내어 다른 용기에 보관하는 것은 의약품의 오용(잘못 사용)에 따른 사고 발생이나 의약품 품질 저하의 원인이 될 수 있으므로 원래의 용기에 넣고 꼭 닫아 보관한다.", " 저장방법": "밀폐용기, 실온보관(1-30℃)", "name_ko": "타이레놀정", "name_en": "mg Tylenol Tab"}', '{"C-Code": "K-004835", "구분": "전문의약품", "제품명": "타모프렉스정20밀리그램(타목시펜시트르산염)", "복지부분류코드": 421.0, "volume": "20밀리그램", "name": "타모프렉스정", "vol_value": 20.0, "vol_unit": "mg", "성분/함량": "[\'Baclofen 바클로펜 20mg\']", "첨가물": "[\'스테아르산마그네슘\', \'옥수수전분\', \'유당수화물\', \'콜로이드성이산화규소\', \'탤크\', \'포비돈\', \'혼합색소파란색(PB-605008)\']", "제형": "정제", "성상": "연한 청색의 원형 정제", "KPIC/ATC": "[\'근골격계/결합조직질환\', \'근이완제\', \'중추성 근이완제\', \'M03BX01\']", "구분(전문/일반)": "전문", "효능": "1. 다발성 경화증, 척추소뇌변성증으로 인한 골격근의 경직 1. 다발성 경화증, 척추소뇌변성증으로 인 한 골격근의 경직 2. 척수질환(염증성, 퇴행성, 외상성, 신생성, 원인불명 등)으로 인한 경직(예 : 경직성 척수마비, 근위축성 축색경화증, 척수공동증, 횡단성척수염, 외상성하지마비, 척수의 압박증, 기타의 척수병 등) 3. 대뇌 원인으로 인한 경직 : 특히 뇌성마비, 뇌혈관사고, 신생성 뇌질환, 퇴행성 뇌질환 등이 있는 경우", "용법": "이 약의 치료는 항상 저용량으로 시작하며, 점차 증량한다. 최적용량은 간 대성경련, 굴근 및 신근의 강직, 경련 등이 감소되고 가능한, 부작용을 피할수 있는 양으로 환자에 따라 개별적으로 설정되어야 한다. 이 약의 치료는 항상 저용량으로 시작하며, 점차 증량한다. 최적용량은 간대성경련, 굴근 및 신근의 강직, 경련 등이 감소되고 가능한, 부작 용을 피할수 있는 양으로 환자에 따라 개별적으로 설정되어야 한다. 과도한 근 위약과 넘어짐을 방지하기 위해, 근 긴장이 직립자세 및 운동의 균형을 유지하는데에 필요하고 또한 기능유지에 사용되는 경우에는 주의하여 사용하도록 한다. 어느 정도의 근긴장이 유지되고 순화 기능이 수행되는데 도움이 될 만한 일시적 연축은 가능하도록 하는 것이 중요하다. 이 약은 소량의 물과 함께 식사중 투여한다 1일 용량은 성인의 경우 1일 3회, 소아의 경우 1일 4회로 분할하여 경구투여한다. ○ 성인 1. 바클로펜으로서 1회 5㎎ 1일 3회 경구투여하고 최적용량이 정해질 때까지 3일간격으로 1일 3회, 1회 5㎎씩 증량한다. 2. 이 약에 과민증 환자는 더 적은 용량(1일 5-10㎎)으로 시작하여 천천히  증량하는 것이 좋다. 3. 최적용량은 보통 1일 30-80㎎이다. 입원환자는 철저한 감독하에 1일 100-120㎎을 투여할 수 있다. ○ 소아(12개월~18세) 1. 치료는 일반적으로 매우 낮은 용량으로 투여한다.(1일 0.3 mg/kg 2~4회 분할 투여) 2. 유지용량의 일반적인 1일 복용량은 0.75~2mg/kg이다. 3. 8세 미만 소아의 1일 최대 복용량은 40mg이고, 8세 이상 소아의 1일 최대 복용량은 60mg이다. 4. 약 1-2주 간격으로 개개인의 요구량에 적합하게 될 때까지 조심스럽게 증량한다. 최대용량 투여이후 6-8주안에 뚜렷한 치료효과가 나타나지 않으면 이 약의 계속 투여여부를 재고한다.", "주의사항": "1. 다음 환자에는 투여하지 말 것. 1) 이 약 및 이 약의 다른 구성성분에 과민증 환자 2) 이 약은 유 당을 함유하고 있으므로, 갈락토오스 불내성(galactose intolerance), Lapp 유당분 해효소 결핍증(Lapp lactase deficiency) 또는 포도당-갈락토오스 흡수장애(glucose-galactose malabsorption) 등의 유전적인 문제가 있는 환자에게는 투여하면 안 된다. 2. 다음 환자에는 신중히 투여할 것 1) 정신병적 장애, 정신분열증, 조울성 장애, 착란상태, 파킨슨병 등이 있는 환자(이 약의 투여로 증상이 악화될 수 있으므 로 신중하게 투여해야 하며 주의 깊은 관찰이 필요하다.) 2) 뇌전증의 병력이 있는 환자에서 이 약의 투여로 경련역치가 낮아질 수 있고, 투여중지 또는 과량투여에 의한 뇌전증발작이 가끔 보고되어 있으므로 신중이 투여한다. 이러한 환자들에 대해서는 적절한 항경련 요법을 유지하고 상태를 충분히 관찰한다. 3) 소화성 궤양 또는 그 병력이 있는 환자 4) 중증의 심부전 환자 5) 심혈관계 질환 환자 6) 뇌혈관계 질환 환자 7) 호흡기계 질환 환자(근이완작용에 의해 호흡억제가 나타날 수 있다.) 8) 간장애 환자 9) 신기능손상 환자(소량을 신중히 투 여한다. 장기적으로 혈액투석을 받는 환자는 혈장 중 이 약의 농도가 상승하므로 1일 약 5mg 정도의 극히 소량을 투여한다.) 10) 이 약의 투여로 배뇨에 영향을 주는 신경성 장애가 개선될 수도 있다. 괄약근에 과도한 긴장이 있었던 환자들에서 급성 요저류가 발생할 수 있으므로 신중이 투여한다. 11) 소아(특히 뇌전증및 그 병력이 있는 환자에서 발작을 유발할 수 있다.) 3. 이상반응 이상반응(예를 들면 진정,  졸음)은 용량을 너무 급속히 증가시켰거나 높은 용량을 투여한 경우 치료 초기에 주로 나타난다. 그 이상반응은 종종 일시적이며 용량을  감소시키면 약화되거나 사라지며 치료를 중단해야 할 정도로 심각한 경우는 드물다. 정신질환의 병력이 있거나 뇌혈관장애(예를 들면 뇌졸중)가 있는 환자, 고령자는 이상반응이 심하게 나타날 수 있다. 이상반응의 빈도에 따라 다음과 같이 분류하였다. : 매우 흔하게 (≥1/10), 흔하게 (≥1/100, <1/10), 흔하지 않게 (≥1/1,000, <1/100), 드물게 (≥1/10,000, <1/1,000), 매우 드물게 (<10,000) 1) 중추신경계 : 매우 흔하게 진정, 졸음, 흔하게 호흡기능억제, 두경감, 권태, 피로, 탈진, 정신혼돈, 어지러움, 두통, 불면증, 다행증, 우울증, 근약화, 운동실조, 진전, 환각, 악몽, 근육통, 안구진탕증, 구갈, 드물게 지각 이상, 구음장애, 경련역치를 낮추어 특히 뇌전증환자에게 경련을 유발할 수 있다. 알코올에 의존적인 환자에서 고용량의 바클로펜(≥100mg)에 의한 중추 수면 무호흡 증후군 사례가 관찰되었다. 2) 감각기계 : 흔하게 시조절장애, 시각장애, 드물게 사시, 미각장애, 이명이 나타날 수 있다. 3) 소화기계 : 매우 흔하게 구역(nausea), 흔하게 경증의 소화기 장애, 구역질(retching), 불수의성 구토, 식욕부진, 위부불쾌감, 복부팽만감, 가슴쓰림, 구갈, 변비, 설사, 드물게 복통, 대변내 잠 혈이 나타날 수 있다. 4) 심혈관계 : 흔하게 저혈압, 심박출량 감소, 심혈관기능 저하, 빈맥, 서맥, 드물게 심계항진, 실신, 흉통이 나타 날 수 있다. 5) 비뇨기계 : 흔하게 빈뇨, 유뇨증, 배뇨곤란, 드물게 요저류, 발기불능, 혈뇨가 나타날 수 있다. 6) 간 : 드물게 간기능 장애, 간기능 검사치이상이 나타날 수 있다. 7) 피부 : 흔하게 다한증, 피부발진이 나타날 수 있다. 8) 기타 : 이 약에 의한 역설적 반응으 로 경직이 증가되었다는 보고가 있었다. 보고된 이상반응 중 대다수는 치료당시의 건강 상태와 관련하여 발생한 것으로 알려져 있다. 때때로 부종, 흉부압박감, 발한, 호흡곤란, 혈당치 상승, 체중증가, 비충혈 등이 나타날 수 있다. 4. 일반적 주의 1) 장기간 사용했던 환자에 서 투여를 갑작스럽게 중단한 경우 불안, 흥분, 착란상태, 환각, 정신병, 조증, 편집상태, 뇌전증중첩상태, 운동이상증, 빈맥, 횡문근융해, 이상고열 및 빈동현상에 의한 일시적인 경직의 악화가 보고되었다. 그러므로 과량투여에 의한 응급사태나 심각한 이상반응이 일어난 경 우 이외에 이 약의 투여를 중지하고자 할 때에는 약 1-2주 이상에 걸쳐 천천히 감량 투여한다. 2) 환자의 반응성에 영향을 주는 현기증,  진정, 졸음 및 주의력 저하작용에 의한 이상반응(졸음 등)이 일어날 수 있으므로 이 약을 투여중인 환자는 자동차 운전 등 위험을 수반하 는 기계조작을 하지 않도록 주의한다. 3) 드물게 혈청중 AST, ALP, 혈당치가 증가되었다는 보고가 있으므로 간장애 환자나 당뇨병 환자는 이 약에 기인하여 위의 수치에 변화가 올 수 있으므로 정기적인 검사를 받아야 한다. 4) 바클로펜은 신기능 손상 환자에서 주의깊게 사용 해야 하며 말기 신부전 환자의 경우 치료상의 유익성이 위험성을 상회한다고 판단되는 경우에만 투여해야 한다. 독성뇌병의 임상 증상(혼 돈, 방향감각상실, 졸음 및 의식의 우울한 정도)을 포함한 과량투여시에 나타나는 신경학적 증상 및 징후가 매일 5mg 이상의 바클로펜 경 구제를 투여한 신기능 손상 환자에서 관찰되었다. 신기능 손상 환자는 초기 독성 증상의 즉각적인 진단을 위해 면밀하게 관찰해야 한다. 5) 이 약으로 치료받은 환자에서 자살 및 자살관련 사례가 보고되었다. 대부분의 사례에서 환자에서 알코올 사용 질환, 우울증 및 이전에  자살 시도 이력을 포함한 자살의 위험성증가와 관련된다. 이 약 치료시 자살에 대한 추가 위험 요인이 있는 환자에 대해 면밀한 감독이 동반되어야 한다. 환자의 가족이나 보호자 또한 환자의 임상적 악화, 자살행동/생각 또는 비정상적인 기분과 행동의 변화에 대해 주의 깊게 관찰하고 이러한 증상 또는 행동이 발현될 경우 즉시 의료전문가에게 보고될 수 있도록 한다. 바클로펜과 관련하여 오용, 남용 및 의존사 례가 보고되었다. 환자가 과거에 약물남용의 경험이 있는지를 확인해야 하고 약물 남용, 오용 및 의존하는 징후(예: 용량증가, 약물추구행동, 내성발현)가 있는지 세심하게 모니터링해야 한다. 5. 상호작용 1) 이 약을 중추신경계에 작용하는 약물, 합성 아편류, 알코올 등과 병용하면 중추신경억제작용(진정작용 등)이 증강될 수 있다. 또 호흡억제의 의험성도 증가되므로 심폐질환이 있는 환자 또는 호흡근이 약화 되어 있는 환자에서 특히 호흡기 및 심혈관 기능에 대하여 충분히 관찰한다. 2) 삼환계 항우울약과 병용 투여시 이 약의 효과가 증강되어 현저한 근긴장저하가 일어날 수 있다. 3) 혈압강하제와 병용투여시 혈압강하작용이 증강될 수 있으므로 혈압강하제의 용량을 적절하게 조 절한다. 모르핀과 및 척수강내에 이 약을 투여를 받는 환자에서 저혈압이 보고된 적이 있다. 4) 이 약과 레보도파를 투여받는 파킨슨병 환자에서 정신혼돈, 환각, 두통, 구역, 격앙이 보고되었다. 6. 임부 및 수유부에 대한 투여 1) 사람에 대한 이 약의 최대 경구투여량(mg/kg)의 13배를 투여한 랫트의 태자에서 배꼴탈장(ventral hernias)의 발생빈도를 증가시키는 것으로 보고되어 있으나 마우스와 토끼에서는 이 와 같은 이상이 나타나지 않았다. 2) 임부에 대하여 확립되어있는 연구가 없고 이 약은 태반을 통과하므로 치료상의 유익성이 태아에 대한 잠재적 위험성을 상회한다고 판단되는 경우에만 투여한다. 3) 이 약의 치료용량에서 유즙으로 활성성분이 매우 소량 분비되므로 유아에서의 이상반응은 없을 것으로 예상된다. 그러나 치료상의 유익성이 위험성을 상회한다고 판단되는 경우에만 투여한다. 7. 고령자에 대한 투 여 고령자에서는 생리기능이 저하되어 있는 경우가 많고 비교적 저용량으로 근력저하, 권태감 등의 증상이 나타날 수 있으므로 저용량부터 투여를 시작하는 등 환자의 상태를 관찰하면서 신중히 투여한다. 8. 과량투여시의 처치 1) 증상 및 징후 : 중추신경억제증상이 특징적으 로 우울증, 졸음, 의식장애, 혼수, 호흡장애, 착란, 환각, 격앙, 경련, 협동운동장애, 시조절장애, 동공반사소실, 전신근긴장저하, 간대성 근경련, 경련, 반사저하증 또는 무반사증, 말초혈관확장, 저혈압, 고혈압, 서맥, 빈맥, 저체온증, 구역, 구토, 설사, 타액분비항진, 간효소치 및 횡문근융해가 증가된다. 중추신경계에 작용하는 약물 또는 물질(예 : 알코올, 디아제팜, 삼환계 항우울약)과 병용한 경우에 이러 한 증상이 악화될 수 있다. 2) 처치 ① 특별한 해독제는 알려져 있지 않다. 위장관으로부터 배출(구토, 위세척, 혼수상태의 환자는 삽관후 위세척), 약용탄 투여. 필요한 경우 염류하제를 사용한다. 호흡억제의 경우 인공호흡과 동시에 심혈관계 기능유지를 위한 처치를 시행한다. ② 저혈압, 고혈압, 경련, 위장관계 장애 및 호흡기, 심혈관계 기능저하 등의 증상에 대한 보조적 처치와 증상 치료를 해야 한다. ③ 독성을 유발할 수 있는 양을 복용한 경우, 특히 복용 후 초기에는 약용탄을 고려한다. 개인별로, 특히 생명을 위협할 수 있는 과량을 복용한  후 초기에는(60분 이내) 위세척(구토, 위세척 등)을 고려해야 한다. 혼수 또는 경련 환자는 위세척을 시작하기 전에 삽관한다. ④ 이 약은 주로 신장으로 배설되므로 가능하면 이뇨제와 함께 충분한 양의 수분을 복용하여야 한다. 신기능 부전이 동반된 심한 중독증에는 투석이  유용할 수 있다. ⑤ 경련이 일어났을 경우에는 디아제팜을 조심스럽게 정맥주사한다.", "저장방법": "밀폐용기, 실온(1~30℃)보관", "name_ko": "프렉스정", "name_en": "Prex Tab"}']

data = package_data + "\n".join(raw_data)

In [48]:
profile = {
'disease': ['아스피린 천식'],
'caution_drugs' : [],
'caution_ingredients' : [],
'current_medications' : ['바르비탈계 약물']
}

user_data = ""
user_data += 'disease: ' + ', '.join(profile.get('disease', [])) or '없음'
user_data += '\ncaution_drugs: ' + ', '.join(profile.get('caution_drugs', [])) or '없음'
user_data += '\ncaution_ingredients: ' + ', '.join(profile.get('caution_ingredients', [])) or '없음'
user_data += '\ncurrent_medications: ' + ', '.join(profile.get('current_medications', [])) or '없음'

system_prompt = f"""
    당신은 약품 성분에 대해서 잘 알고 있는 약사 AI 입니다.
    [사용자 정보]를 통해 사용자가 가지고 있는 병과, 주의해야하는 알약, 주의해야하는 성분을 확인하고
    [사전 정보]와 비교하여 [입력정보]로 들어오는 약이 섭취 가능한지 판단해야합니다.

    [판단 방법]
    1단계: [사용자 정보]의 'caution_drugs'이 해당 약품이 포함되는 경우 섭취 불가능
    2단계: [사용자 정보]의 'caution_ingredients'이 약품의 [제품명] 또는 [성분/함량] 또는 [첨가물]에 포함되는 경우 섭취 불가능
    3단계: [사용자 정보]의 'disease'의 질병을 가지고 있는 경우 해당 약품의 [주의사항]에서 해당 질병이 있을 경우 주의해야한다는 설명이 있을 경우 섭취 불가능
    4단계: [사용자 정보]의 'current_medications'과의 상호작용 가능성이 있는지 확인, 부정적 상호작용이 있을 경우 섭취 불가능
    * 1-4 단계를 모두 통과할 경우 섭취 가능하다고 판단한다
    * 명확하게 섭취가 불가능한 경우가 아닌경우는 섭취 가능하다고 판단한다
    
    [사용자 정보]
    {user_data}

    [사용자 정보 설명]
    없음이라고 작성되었다면 해당 질병이 없거나, 주의한 성분이 없다는 뜻이므로 무시한다.
    disease: 사용자가 가지고 있는 질병 명들
    caution_drugs : 사용자가 주의해야하는 약품 명들 (주의해야하는 약품과 비슷한 성분이 있을 경우 미섭취 권장)
    caution_ingredients : 사용자가 섭취할 수 없는 주의해야하는 성분, 첨가물 등 (주의해야하는 성분이 있을 경우 미섭취 권장)
    current_medications : 사용자가 복용 중인 약
    
    [사전 정보]
    {data}
    
    [입력 정보]
    섭취하려는 약에 대한 이름

    [출력 형식]
    {{
        "name" : "제품명",
        "effect" : "이 약의 효능",
        "isUsable" : "섭취 가능 여부를 Bool 값으로",
        "unusable_reason" : "[사용자 정보]를 바탕으로 섭취가 불가능한 이유를 설명합니다, 섭취 가능한 경우 빈칸으로 제공합니다",
        "cautionary_ingredients" : "[사용자 정보]에서 섭취 불가능한 이유가 되는 주의 성분을 리스트로 제공합니다, 섭취 가능한 경우 빈칸으로 제공합니다",
        "caution" : "이 약품의 일반적인 주의사항을 제공합니다"
    }}
    """

In [ ]:
# import json
# facts = {
#     '제품명': '타이레놀정', 
#     '성분/함량': [['아세트아미노펜(USP)', '50Mg']], 
#     '첨가물': ['옥수수전분', '전분', '셀룰로오스', '스테아르산마그네슘'], 
#     '제형/성상': '백색의 장방형필름코팅정제', 
#     'KPIC/ATC': '없음', 
#     '구분': '없음', 
#     '효능': ['감기로인한발열및동통(통 증)', '두통', '신경통', '근육통', '월경통', '염좌통(뻔통증)', '치통', '관절통', '류마티양동통(통증이상 또는그병력)'], 
#     '용법': ['12세이상 소아및성인 1회 1-2정씩 1일 3-4회(4-6시간마다)최대 8정(4a)를 초과하여 복용하지 말 것'], 
#     '주의사항': ['삼환계항우 울제·알코올을복용한사람', '심한심장기능저하환자', '아스피린천식(비스테로있는환자)', '바르비탈계약물', '이드성소염(항염)제에의한천식발작유발', '유효성분 1정중아세트아 미노펜(USP)…50Mg', '효능효과감기로인한발열및동통(통증)두통,신경통근육통,월경통,염좌통(뻔통증)치통,관절통,류마티양동통(통증이상 또는그병력)', '출혈경향이있는환자', '간장애 또는그병력', '신장(콩팥)장애 또는그병력', '소화성궤양의병력', '혈액', '복용 시 주의할사항', '복용 후속환자', '과민증의병력있는환자', '기관지천식환자', '고령자(노인)', '임부/수유부', '파린장기복용환자(혈소판기능이상)', '심장기능이상환자', '복용 하여서는안됨', '아세트아미노펜을포함하는다른제품과함께', '약사와상의할것', '이약투 여후피부발진이나다른이약', '일일최대용량(4000Mg)을초과하여복용시간손상', '매일세잔 이상정기적술복용시주의', '과민반응', '즉시복용중단', '아세트아미노펜함유', '최다손 이닿지않는곳에보관', '치과의사약사와상의할것']}
# package_data = json.dumps(facts, ensure_ascii=False)

user = '타이레놀정'

In [49]:
answer = chat(system_prompt, '타이레놀정')
print(answer)

{
    "name" : "타이레놀정",
    "effect" : "감기로 인한 발열 및 동통(통증), 두통, 신경통, 근육통, 월경통, 염좌통(뻐근통증), 치통, 관절통, 류마티양동통(통증 이상 또는 그 병력)",
    "isUsable" : false,
    "unusable_reason" : "사용자가 가지고 있는 질병인 아스피린 천식이 있어, 이 약의 주의사항에 명시된 ‘아스피린천식(비스테로이드성 소염제에 의한 천식발작 유발)’에 해당하여 섭취하면 안 됩니다. 또한, 바르비탈계 약물을 복용 중이므로, 이 약과의 상호작용 가능성도 고려하여 섭취를 권장하지 않습니다.",
    "cautionary_ingredients" : [],
    "caution" : "이 약은 감기 증상에 대한 일반적인 진통제이며, 과량 복용 시 간손상 위험이 있으니 용법과 용량을 준수하고, 이상반응이 발생하거나 증상이 지속되면 의사와 상담하십시오."
}
